In [ ]:
# Imports and configuration
import os
import math
import requests
import json
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from urllib.parse import parse_qs, urlparse
import sys

API_BASE_URL = "http://localhost:3000/api/v1/lakehouse/network-analysis/transaction/"
FALLBACK_ACCOUNT_ID = "dbtrAcct_24a03dafa2c14f6da6bfc195d57c6d21"

def fetch_transaction_network(account_id):
    url = f"{API_BASE_URL}{account_id}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.json()
    except Exception:
       
        return None

def get_account_id_from_url():
    
    if len(sys.argv) > 0 and '?' in sys.argv[0]:
        query = urlparse(sys.argv[0]).query
        params = parse_qs(query)
        return params.get('accountId', [FALLBACK_ACCOUNT_ID])[0]
    return os.getenv('ACCOUNT_ID', FALLBACK_ACCOUNT_ID)


In [ ]:

account_id = get_account_id_from_url()
data = fetch_transaction_network(account_id)


In [ ]:
if not data:
    display(HTML(f"<div style='color:#ef4444'>No network data available for account: {account_id}</div>"))
else:
    
    nodes = []
    edges = []
    
    center = data.get('centerAccount')
    if center:
        nodes.append({
            'id': center.get('accountId'),
            'label': center.get('accountHolder'),
            'type': 'center',
            'status': 'normal', 
            'isCenter': True
        })
        
    for acc in data.get('connectedAccounts', []):
        nodes.append({
            'id': acc.get('accountId'),
            'label': acc.get('accountHolder'),
            'type': 'connected',
            'status': 'alert' if acc.get('hasAlert') else 'normal',
            'isCenter': False
        })
        
    edges = data.get('edges', [])
    
  
    positions = {}
    center_idx = 0
  
    for i, n in enumerate(nodes):
        if n.get('isCenter'):
            center_idx = i
            break
            
   
    radius = 300
    n_nodes = len(nodes)
    angle_step = 2 * math.pi / max(n_nodes - 1, 1)
    j = 0
    for i, node in enumerate(nodes):
        if i == center_idx:
            positions[node['id']] = (0.0, 0.0)
        else:
            angle = angle_step * j
            positions[node['id']] = (radius * math.cos(angle), radius * math.sin(angle))
            j += 1

    edge_x = []
    edge_y = []
    for e in edges:
        s = e.get('source')
        t = e.get('target')
        if s in positions and t in positions:
            x0, y0 = positions[s]
            x1, y1 = positions[t]
            edge_x += [x0, x1, None]
            edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, 
        mode='lines', 
        line=dict(width=1, color='#cbd5e1', dash='solid'), 
        hoverinfo='none'
    )

    node_x = []
    node_y = []
    node_text = []
    marker_size = []
    marker_color = []
    node_ids = []

    for n in nodes:
        nid = n.get('id')
        if nid not in positions: continue
        x, y = positions[nid]
        node_x.append(x)
        node_y.append(y)
        label = n.get('label') or nid
        node_text.append(f"{label}")
        
       
        is_center = n.get('isCenter')
        status = n.get('status')
        
        if is_center:
            color = '#f59e0b' 
            size = 32
        elif status == 'alert':
            color = '#ef4444' 
            size = 22
        else:
            color = '#6366f1' 
            size = 22
            
        marker_color.append(color)
        marker_size.append(size)
        node_ids.append(nid)

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode='markers+text',
        text=[t.split(' ')[0] for t in node_text], 
        hovertext=node_text,
        hoverinfo='text',
        customdata=node_ids,
        marker=dict(
            showscale=False,
            color=marker_color,
            size=marker_size,
            line_width=2,
            line_color='#ffffff'
        ),
        textposition='bottom center'
    )

    fig = go.Figure(
        data=[edge_trace, node_trace],
        layout=go.Layout(
            title=None,
            showlegend=False,
            hovermode='closest',
            margin=dict(b=20,l=5,r=5,t=20),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            height=600,
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)',
        )
    )

    graph_id = 'tx-network-graph'
    graph_html = pio.to_html(fig, full_html=False, config={'displayModeBar': False}, div_id=graph_id)
    
    sidebar_html = """
    <div id="sidebar-container" style="width: 240px;  color: #0f172a; padding: 24px; border-radius: 24px;  font-family: 'Inter', system-ui, sans-serif; margin-left: 24px;flex-direction: column; height: 600px; overflow-y: auto; transition: all 0.3s ease;">
        <!-- OVERVIEW STATE -->
        <div id="view-overview">
            <h2 style="margin: 0 0 20px 0; color: #0f172a; font-size: 1.25rem; font-weight: 700;  padding-bottom: 12px;">Transaction Network</h2>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Center Account</div>
                <div id="overview-id" style="font-weight: 600; font-size: 1rem; color: #0f172a;">-</div>
            </div>
            
            <div style="margin-bottom: 24px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Holder</div>
                <div id="overview-name" style="font-weight: 600; font-size: 1.1rem; color: #0284c7;">-</div>
            </div>
            
            <div style="background: #f8fafc; padding: 20px; ">
                <h3 style="margin-top: 0; font-size: 0.85rem; color: #64748b; margin-bottom: 16px; text-transform: uppercase; font-weight: 600;">Summary</h3>
                <div style="display: grid; grid-template-columns: 1fr; gap: 16px;">
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Connected Accounts</span>
                        <span id="overview-connected" style="font-weight: 600; color: #0f172a;">0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Total Edges</span>
                        <span id="overview-edges" style="font-weight: 600; color: #0f172a;">0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Alerts</span>
                        <span id="overview-alerts" style="font-weight: 600; color: #dc2626;">0</span>
                    </div>
                </div>
            </div>
        </div>

        <!-- DETAILS STATE -->
        <div id="view-details" style="display: none;">
            <h2 style="margin: 0 0 20px 0; color: #0f172a; font-size: 1.25rem; font-weight: 700;  padding-bottom: 12px;">Node Details</h2>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Account ID</div>
                <div id="side-id" style="font-weight: 600; font-size: 1rem; color: #0f172a; word-break: break-all;">-</div>
            </div>
            
            <div style="margin-bottom: 16px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Holder</div>
                <div id="side-name" style="font-weight: 600; font-size: 1.1rem; color: #d97706;">-</div>
            </div>
            
            <div style="margin-bottom: 24px;">
                <div style="font-size: 0.7rem; color: #64748b; text-transform: uppercase;">Status</div>
                <div id="side-status" style="display: inline-block; padding: 4px 12px; border-radius: 9999px; font-size: 0.75rem; margin-top: 4px; font-weight: 500;">-</div>
            </div>

            <div style="background: #f8fafc; padding: 20px; border-radius: 20px;">
                <h3 style="margin-top: 0; font-size: 0.85rem; color: #64748b; margin-bottom: 16px; text-transform: uppercase; font-weight: 600;">Connections</h3>
                <div style="display: grid; grid-template-columns: 1fr; gap: 16px;">
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Inbound</span>
                        <span id="side-inbound" style="font-weight: 600; color: #059669;">0</span>
                    </div>
                    <div style="display: flex; justify-content: space-between;">
                        <span style="color: #64748b; font-size: 0.8rem;">Outbound</span>
                        <span id="side-outbound" style="font-weight: 600; color: #0f172a;">0</span>
                    </div>
                </div>
            </div>
            
            <div style="margin-top: 24px; text-align: center;">
                <button onclick="resetSidebar()" style="background: white; border: 1px solid #cbd5e1; color: #64748b; padding: 8px 16px; border-radius: 8px; font-size: 0.75rem; cursor: pointer; font-weight: 500; transition: all 0.2s;">Back to Overview</button>
            </div>
        </div>
    </div>
    """
    
    #
    interactive_script = f"""
    <script>
    (function() {{
        const nodes = {json.dumps(nodes)};
        const edges = {json.dumps(edges)};
        const centerAcct = {json.dumps(center)};
        const graphDiv = document.getElementById('{graph_id}');
        
        function initOverview() {{
            if(centerAcct) {{
                document.getElementById('overview-id').innerText = centerAcct.accountId || '-';
                document.getElementById('overview-name').innerText = centerAcct.accountHolder || '-';
            }}
            
            const connectedCount = nodes.filter(n => !n.isCenter).length;
            const alertCount = nodes.filter(n => n.status === 'alert').length;
            
            document.getElementById('overview-connected').innerText = connectedCount;
            document.getElementById('overview-edges').innerText = edges.length;
            document.getElementById('overview-alerts').innerText = alertCount;
        }}
        
        window.resetSidebar = function() {{
            document.getElementById('view-overview').style.display = 'block';
            document.getElementById('view-details').style.display = 'none';
            initOverview();
        }};
        
        function updateSidebar(nodeId) {{
            const node = nodes.find(n => n.id === nodeId);
            if (!node) return;
            
            document.getElementById('view-overview').style.display = 'none';
            document.getElementById('view-details').style.display = 'block';
            
            document.getElementById('side-id').innerText = node.id;
            document.getElementById('side-name').innerText = node.label || '-';
            
            const statusEl = document.getElementById('side-status');
            statusEl.innerText = node.status === 'alert' ? 'ALERT' : 'NORMAL';
            statusEl.style.background = node.status === 'alert' ? '#fef2f2' : '#f0f9ff';
            statusEl.style.color = node.status === 'alert' ? '#dc2626' : '#0369a1';
            
            let inbound = 0, outbound = 0;
            edges.forEach(e => {{
                if(e.source === nodeId) outbound++;
                if(e.target === nodeId) inbound++;
            }});
            
            document.getElementById('side-inbound').innerText = inbound;
            document.getElementById('side-outbound').innerText = outbound;
        }}
        
        setTimeout(initOverview, 500);
        
        if (graphDiv) {{
            graphDiv.on('plotly_click', d => {{
                if(d.points && d.points[0] && d.points[0].customdata) {{
                    updateSidebar(d.points[0].customdata);
                }}
            }});
            graphDiv.on('plotly_doubleclick', resetSidebar);
        }}
    }})();
    </script>
    """
    
    container_html = f"""
    <div style="display: flex; align-items: stretch; background-color: #ffffff; padding: 24px;  gap: 24px;">
        <div style="flex: 1; background: #f8fafc; border-radius: 24px;  overflow: hidden; position: relative;">
            {graph_html}
        </div>
        {sidebar_html}
    </div>
    {interactive_script}
    """
    display(HTML(container_html))


In [ ]:

import os
from urllib.parse import parse_qs, urlparse
from IPython.display import display, HTML
import sys

def get_account_id_from_url():
    if 'VOILA_BASE_URL' in os.environ:
        if len(sys.argv) > 0 and '?' in sys.argv[0]:
            query = urlparse(sys.argv[0]).query
            params = parse_qs(query)
            return params.get('accountId', [FALLBACK_ACCOUNT_ID])[0]
   
    return FALLBACK_ACCOUNT_ID

account_id = get_account_id_from_url()
transaction_data = fetch_transaction_network(account_id)



In [ ]:
import networkx as nx
import plotly.graph_objects as go

def build_network_graph(data):
    G = nx.DiGraph()
    
    center = data['centerAccount']
    G.add_node(center['accountId'], label=center['accountHolder'], type='center', hasAlert=False)

    for acc in data.get('connectedAccounts', []):
        G.add_node(acc['accountId'], label=acc['accountHolder'], type='connected', hasAlert=acc['hasAlert'])
  
    for edge in data.get('edges', []):
        G.add_edge(edge['source'], edge['target'], type=edge['type'])
    return G

if transaction_data:
    G = build_network_graph(transaction_data)
else:
    G = None

In [ ]:

try:
    if 'plot_network' in globals() and callable(plot_network):
        plot_network(G, transaction_data)
    else:
        from IPython.display import HTML, display
        display(HTML("<div style='color:#f59e0b'>Plot function unavailable.</div>"))
except Exception:
    from IPython.display import HTML, display
    display(HTML("<div style='color:#ef4444'>An error occurred rendering the plot.</div>"))